# 02 - Exploratory Data Analysis (Phase 2)

This notebook performs EDA directly from the raw training dataset `data/train_test.csv`.

Outputs:
- figures saved to `figures/eda/`
- supporting markdown reports generated by this notebook run (see later cells)

No preprocessing, feature engineering, or modeling is performed here.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(".").resolve()
DATA_PATH = PROJECT_ROOT / "data/train_test.csv"
FIG_DIR = PROJECT_ROOT / "figures" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.shape, df.columns.tolist()

In [ ]:
# Basic column typing
dtypes = df.dtypes.astype(str)
dtypes

In [ ]:
# Identify likely target and feature groups using column names only.
target_col = "posted_rate" if "posted_rate" in df.columns else None
load_id_col = "load_id" if "load_id" in df.columns else None

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = [c for c in df.columns if c not in numeric_cols]

target_col, load_id_col, len(numeric_cols), len(object_cols)

## Target variable analysis

In [ ]:
if target_col is None:
    raise ValueError("Target column 'posted_rate' not found in training data.")

target_desc = df[target_col].describe()
target_missing = df[target_col].isna().sum()
target_desc, target_missing

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df[target_col].dropna().values, bins=50, color="#064A56", alpha=0.85)
plt.title("Target distribution: posted_rate")
plt.xlabel("posted_rate")
plt.ylabel("count")
plt.grid(axis="y", alpha=0.25)
out = FIG_DIR / "posted_rate_hist.png"
plt.tight_layout()
plt.savefig(out, dpi=180)
plt.close()
out

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(df[target_col].dropna().values, vert=True, showfliers=True)
plt.title("Target boxplot: posted_rate")
plt.ylabel("posted_rate")
out = FIG_DIR / "posted_rate_box.png"
plt.tight_layout()
plt.savefig(out, dpi=180)
plt.close()
out

## Numerical feature analysis

In [ ]:
numeric_summary = df[numeric_cols].describe().T
numeric_summary

In [ ]:
# Plot key numeric columns (if present)
key_numeric_candidates = [
    c for c in [
        "distance", "weight", "market_index", "quote_signal",
        "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"
    ] if c in df.columns
]

for c in key_numeric_candidates:
    plt.figure(figsize=(10, 4))
    plt.hist(df[c].dropna().values, bins=50, color="#064A56", alpha=0.85)
    plt.title(f"Histogram: {c}")
    plt.xlabel(c)
    plt.ylabel("count")
    plt.grid(axis="y", alpha=0.25)
    out = FIG_DIR / f"{c}_hist.png"
    plt.tight_layout()
    plt.savefig(out, dpi=180)
    plt.close()

sorted([p.name for p in FIG_DIR.glob('*.png')])[:10]

## Categorical feature analysis

In [ ]:
categorical_cols = [c for c in object_cols if c not in {load_id_col, "date"}]
categorical_cols

In [ ]:
cat_summary = []
for c in categorical_cols:
    s = df[c].astype("string")
    cat_summary.append({
        "feature": c,
        "n_unique": s.nunique(dropna=True),
        "n_missing": int(s.isna().sum()),
        "top_10": s.value_counts(dropna=True).head(10).to_dict(),
    })
cat_summary

In [ ]:
# Bar charts for top categories (excluding load_id)
for c in categorical_cols:
    vc = df[c].astype("string").value_counts(dropna=True).head(15)
    plt.figure(figsize=(10, 4))
    vc.plot(kind="bar", color="#064A56", alpha=0.85)
    plt.title(f"Top categories: {c}")
    plt.xlabel(c)
    plt.ylabel("count")
    plt.tight_layout()
    out = FIG_DIR / f"{c}_top_categories.png"
    plt.savefig(out, dpi=180)
    plt.close()
out

## Missing value analysis

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df) * 100.0).round(4)
missing_df = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
missing_df[missing_df.missing_count > 0]

In [ ]:
missing_df_plot = missing_df[missing_df.missing_count > 0].head(30)
if not missing_df_plot.empty:
    plt.figure(figsize=(10, 4))
    plt.bar(missing_df_plot.index.astype(str), missing_df_plot["missing_pct"], color="#064A56", alpha=0.85)
    plt.xticks(rotation=45, ha='right')
    plt.title("Missingness % by feature")
    plt.ylabel("missing %")
    plt.tight_layout()
    out = FIG_DIR / "missingness_pct.png"
    plt.savefig(out, dpi=180)
    plt.close()
    out

In [ ]:
# Use IQR-based outlier detection as descriptive analysis.
outlier_rows = []
for c in numeric_cols:
    x = df[c].dropna().astype(float)
    if len(x) < 10:
        continue
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_count = int(((x < lo) | (x > hi)).sum())
    outlier_pct = 100.0 * outlier_count / len(x)
    outlier_rows.append({"feature": c, "outlier_count": outlier_count, "outlier_pct": outlier_pct, "lo": lo, "hi": hi})

outliers_df = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
outliers_df.head(20)

In [ ]:
# Boxplots for key numeric columns
for c in key_numeric_candidates + ([target_col] if target_col in df.columns else []):
    plt.figure(figsize=(8, 4))
    plt.boxplot(df[c].dropna().values, vert=True, showfliers=True)
    plt.title(f"Boxplot: {c}")
    plt.ylabel(c)
    plt.tight_layout()
    out = FIG_DIR / f"{c}_box.png"
    plt.savefig(out, dpi=180)
    plt.close()
sorted([p.name for p in FIG_DIR.glob('*box*.png')])[:5]

## Correlation analysis (numerical)

In [ ]:
corr = df[numeric_cols].corr(numeric_only=True)
corr

In [ ]:
plt.figure(figsize=(10, 8))
im = plt.imshow(corr.values, cmap="viridis", aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Correlation matrix (numerical features)")
plt.tight_layout()
out = FIG_DIR / "correlation_matrix.png"
plt.savefig(out, dpi=180)
plt.close()
out

## Geographic analysis

In [ ]:
geo_candidates = [c for c in ["pickup_lat","pickup_lon","delivery_lat","delivery_lon"] if c in df.columns]
geo_candidates

In [ ]:
for lat_col, lon_col, label in [
    ("pickup_lat","pickup_lon","pickup"),
    ("delivery_lat","delivery_lon","delivery")
]:
    if lat_col in df.columns and lon_col in df.columns:
        plt.figure(figsize=(6, 4))
        plt.scatter(df[lon_col], df[lat_col], s=1, alpha=0.25)
        plt.title(f"Geo scatter: {label}")
        plt.xlabel(lon_col)
        plt.ylabel(lat_col)
        plt.tight_layout()
        out = FIG_DIR / f"geo_scatter_{label}.png"
        plt.savefig(out, dpi=180)
        plt.close()
        
sorted([p.name for p in FIG_DIR.glob('geo_scatter_*.png')])

## Temporal analysis

In [ ]:
if "date" not in df.columns:
    raise ValueError("Expected 'date' column for temporal analysis.")

date_parsed = pd.to_datetime(df["date"], errors="coerce")
df["date_parsed"] = date_parsed
date_parsed.isna().sum(), date_parsed.min(), date_parsed.max()

In [ ]:
if date_parsed.notna().any():
    daily = df.loc[df["date_parsed"].notna()].groupby("date_parsed")[target_col].mean().sort_index()
    plt.figure(figsize=(10, 4))
    plt.plot(daily.index, daily.values, color="#064A56")
    plt.title("Mean posted_rate by date")
    plt.xlabel("date")
    plt.ylabel("mean posted_rate")
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    out = FIG_DIR / "mean_posted_rate_by_date.png"
    plt.savefig(out, dpi=180)
    plt.close()
    out

## Business relationship analysis

In [ ]:
# Group-by target relationships.
group_insights = []
for c in ["pickup","delivery","equipment"]:
    if c in df.columns:
        tmp = df.groupby(c)[target_col].agg(['count','mean','median']).sort_values('mean', ascending=False)
        group_insights.append((c, tmp.head(10)))

[(c, t.shape) for c,t in group_insights]

In [ ]:
group_pickup = group_insights[0][1] if group_insights else None
group_pickup

## Feature engineering opportunities (from EDA observations)

In [ ]:
# Simple heuristic proposals based on detected column types.
# Note: final decisions for Phase 3 must be implemented later, using these EDA findings.
opps = []
if "date" in df.columns:
    opps.append("Derive calendar/time features from date (year/month/day/weekday/weekend).")

for c in ["pickup","delivery","equipment"]:
    if c in df.columns:
        opps.append(f"Encode categorical feature {c}.")

if all(c in df.columns for c in ["weight","distance"]):
    opps.append("Create weight-per-distance interaction feature (weight/distance).")
if all(c in df.columns for c in ["market_index","quote_signal"]):
    opps.append("Create interactions between market_index and quote_signal if shown correlated with target.")

opps

## Modeling implications (descriptive only)

In [ ]:
modeling_implications = []
if missing_df[missing_df.missing_count > 0].shape[0] > 0:
    modeling_implications.append("Missing values exist (see missingness analysis); require imputation policy before modeling.")
if outliers_df.shape[0] > 0:
    modeling_implications.append("Outliers detected by IQR; consider robust scaling and/or outlier-aware preprocessing.")
if categorical_cols:
    modeling_implications.append("Categorical variables with moderate cardinality; choose encoding strategy (likely one-hot) mindful of dimensionality.")
if date_parsed.notna().any():
    modeling_implications.append("Date column available; consider time-based features capturing seasonality/temporal effects.")

modeling_implications

In [ ]:
# Write markdown reports
def df_to_md_table(d: pd.DataFrame, max_rows: int = 25) -> str:
    if d is None or d.empty:
        return "- (empty)"
    head = d.head(max_rows)
    return head.to_markdown()

missing_section = missing_df[missing_df.missing_count > 0].to_markdown() if (missing_df.missing_count > 0).any() else "- None detected"
outliers_section = outliers_df.head(20).to_markdown(index=False) if not outliers_df.empty else "- None"
corr_with_target = None
if target_col in corr.columns:
    corr_with_target = corr[target_col].sort_values(ascending=False)

corr_section = corr_with_target.to_frame(name="corr").reset_index().rename(columns={'index':'feature'}).head(20).to_markdown(index=False) if corr_with_target is not None else "- n/a"

eda_report = []
eda_report.append("# Exploratory Data Analysis (Phase 2) - data/train_test.csv\n")
eda_report.append(f"**Shape:** {df.shape[0]:,} rows x {df.shape[1]:,} columns\n")
eda_report.append("## Target variable analysis\n")
eda_report.append(f"**Target column:** `{target_col}`\n")
eda_report.append(target_desc.to_frame(name="value").to_markdown())
eda_report.append("\n## Missing value analysis\n")
eda_report.append(missing_section)
eda_report.append("\n## Numerical feature summary\n")
eda_report.append(numeric_summary.head(20).to_markdown())
eda_report.append("\n## Outlier analysis (IQR-based)\n")
eda_report.append(outliers_section)
eda_report.append("\n## Correlation analysis (top corr with target)\n")
eda_report.append(corr_section)
eda_report.append("\n## Categorical feature analysis\n")
for c in categorical_cols:
    s = df[c].astype("string")
    vc = s.value_counts(dropna=True).head(10)
    eda_report.append(f"### {c}\n")
    eda_report.append(vc.to_frame(name="count").to_markdown())

eda_report.append("\n## Geographic analysis\n")
eda_report.append("See figures in `figures/eda/` for geo scatters and histograms.\n")
eda_report.append("\n## Temporal analysis\n")
eda_report.append(f"Date parse: missing={int(df['date_parsed'].isna().sum())}, min={date_parsed.min()}, max={date_parsed.max()}\n")
eda_report.append("\n## Data quality observations\n")
eda_report.append("- Missingness present: see missingness section.\n")
eda_report.append("- Outliers present by IQR rule.\n")
eda_report.append("- Date parsing errors (if any) are captured in temporal analysis.\n")
eda_report.append("\n## Feature engineering opportunities\n")
for o in opps:
    eda_report.append(f"- {o}\n")

eda_report.append("\n## Modeling implications\n")
for mi in modeling_implications:
    eda_report.append(f"- {mi}\n")

eda_text = "\n".join(eda_report)

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
out_eda = REPORTS_DIR / "exploratory_data_analysis.md"
out_eda.write_text(eda_text, encoding="utf-8")
out_eda

In [ ]:
business_insights = []
business_insights.append("# Business Insights (Phase 2) - from EDA\n")
business_insights.append(f"**Target:** {target_col}\n")

business_insights.append("## Interpretable patterns to validate in Phase 3\n")
business_insights.append("- Confirm whether distance and weight show strong relationship to posted_rate and whether robust scaling is appropriate.")
business_insights.append("- Check whether temporal effects (date-derived features) correlate with shifts in posted_rate.")
business_insights.append("- Inspect categorical groups (pickup/delivery/equipment) for stable differences in average posted_rate.")

insight_text = "\n".join(business_insights)
out_bi = REPORTS_DIR / "business_insights.md"
out_bi.write_text(insight_text, encoding="utf-8")
out_bi